<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 디렉토리 경로 설정
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model/data'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')

os.makedirs(PROCESSED_DIR, exist_ok=True)

# 제약 조건 설정 변수 (수정됨: 방향성 피크 임계값으로 대체)
PHONE_Z_THRESHOLD = 10.0  # 스마트폰 제트축 양수 피크 임계값
SPEN_Y_THRESHOLD = -5.0   # 에스펜 와이축 음수 피크 임계값
WINDOW_SIZE = 20  # 200ms 윈도우 내의 대략적인 데이터 샘플 수 (50Hz 기준 10~20개)

In [ ]:
def parse_semicolon_data(data_string):
    """세미콜론으로 구분된 문자열을 실수형 넘파이 배열로 파싱"""
    if pd.isna(data_string):
        return np.zeros(WINDOW_SIZE)
    try:
        return np.array([float(x) for x in str(data_string).split(';') if x.strip() != ''])
    except ValueError:
        return np.zeros(WINDOW_SIZE)

In [ ]:
def process_raw_files():
    processed_features = []
    processed_labels = []

    # 1. raw 폴더의 모든 CSV 파일 순회
    for filename in os.listdir(RAW_DIR):
        if not filename.endswith('.csv'):
            continue

        filepath = os.path.join(RAW_DIR, filename)
        df = pd.read_csv(filepath)

        # 2. 윈도우 프레임 단위 처리
        for index, row in df.iterrows():
            label = row['label']

            # 각 축의 센서 데이터 파싱 (길이가 다를 경우 패딩이나 자르기 처리 필요)
            accel_x = parse_semicolon_data(row['phoneAccelX'])
            accel_y = parse_semicolon_data(row['phoneAccelY'])
            accel_z = parse_semicolon_data(row['phoneAccelZ'])
            gyro_x = parse_semicolon_data(row['phoneGyroX'])
            gyro_y = parse_semicolon_data(row['phoneGyroY'])
            gyro_z = parse_semicolon_data(row['phoneGyroZ'])
            spen_x = parse_semicolon_data(row['spenDeltaX'])
            spen_y = parse_semicolon_data(row['spenDeltaY'])

            # 3. 다중 피크 처리 규칙 및 방향성 임계값 검사 (수정됨)
            # 200ms 윈도우 내에서 스마트폰 제트축의 최댓값과 에스펜 와이축의 최솟값 탐색
            max_z_accel = np.max(accel_z) if len(accel_z) > 0 else 0
            min_y_spen = np.min(spen_y) if len(spen_y) > 0 else 0

            # 제트축이 양수 임계값을 넘거나, 와이축이 음수 임계값보다 낮을 때 유효 타격으로 인정
            if max_z_accel >= PHONE_Z_THRESHOLD or min_y_spen <= SPEN_Y_THRESHOLD:
                # 8축 데이터를 (8, WINDOW_SIZE) 형태의 행렬로 결합
                feature_matrix = np.vstack([
                    accel_x[:WINDOW_SIZE], accel_y[:WINDOW_SIZE], accel_z[:WINDOW_SIZE],
                    gyro_x[:WINDOW_SIZE], gyro_y[:WINDOW_SIZE], gyro_z[:WINDOW_SIZE],
                    spen_x[:WINDOW_SIZE], spen_y[:WINDOW_SIZE]
                ])

                processed_features.append(feature_matrix)
                processed_labels.append(label)

    return np.array(processed_features), np.array(processed_labels)

In [ ]:
# 전처리 실행
print("데이터 전처리를 시작합니다...")
features, labels = process_raw_files()

# 4. 텐서 형태의 학습용 데이터셋으로 저장
features_tensor = torch.tensor(features, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.float32) # 라벨 인코딩 로직 추가 필요

torch.save(features_tensor, os.path.join(PROCESSED_DIR, 'features.pt'))
torch.save(labels_tensor, os.path.join(PROCESSED_DIR, 'labels.pt'))

print(f"전처리 완료. 유효 타격 샘플 수: {len(features)}")
print(f"데이터가 {PROCESSED_DIR}에 저장되었습니다.")